## Travel Mode Dataset (NTS0409)

In this section, the National Travel Survey (NTS0409) dataset is used to construct the travel mode behaviour component of the framework.

The NTS0409 dataset provides the average number of trips per person per year, disaggregated by trip purpose and travel mode. Specifically, it records how frequently different travel modes (e.g., walking, car driver, bus) are used for each trip purpose (e.g., commuting, shopping, leisure).

### Purpose of Using This Dataset

The objective of using this dataset is to derive the conditional probability distribution of travel modes given a trip purpose:

P (Travel Mode | Trip Purpose)

Within the proposed framework, the trip purpose is first determined based on demographic-specific mobility patterns. Given this trip purpose, the NTS0409 dataset is then used to estimate how individuals choose their travel mode.

### Data Processing Overview

Since the dataset reports annual trip frequencies rather than probabilities, a normalisation step is required. For each trip purpose, the number of trips across all travel modes is summed, and each mode-specific value is divided by this total. This converts the raw trip counts into a probability distribution over travel modes.

As a result, the final output of this step is a travel mode probability matrix, where each row corresponds to a trip purpose and each row sums to 1.

### Step 1: Import dataset

In [1]:
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [2]:
nts0409a = pd.read_excel('../dataset(mobility)/nts0409-Average number of trips and distance travelled by purpose and main mode England, 2002 onwards.ods', sheet_name='NTS0409a_trips', skiprows=5)

In [3]:
display(nts0409a.head())

,Year,Main mode,Commuting,Business,Education or escort education,Shopping,Other escort,Personal business,Leisure [note 2],Just walk [note 8],Other [note 8],All purposes,Unweighted sample size: trips (thousands)
0,2002,"Walk [notes 3, 4]",16.349172,2.865760,44.115380,59.249569,13.319239,34.740433,52.424669,40.874737,0.073657,264.012617,66.356
1,2002,Pedal cycle [note 5],6.547287,0.612595,1.833587,2.363763,0.364849,1.308123,4.999214,0.000000,0.000000,18.029417,4.542
2,2002,Car or van driver,95.832523,26.226708,23.895374,90.122423,57.491243,47.131800,97.393661,0.000000,0.320400,438.414132,116.736
3,2002,Car or van passenger,16.985655,2.400159,25.759621,46.019618,30.738702,24.580872,93.407171,0.000000,0.043487,239.935284,63.814
4,2002,Motorcycle,2.005321,0.041155,0.051922,0.471992,0.114125,0.297173,0.864034,0.000000,0.006289,3.852011,0.977


In [4]:
nts0409a.columns

Index(['Year', 'Main mode', 'Commuting', 'Business',
       'Education or escort education', 'Shopping', 'Other escort',
       'Personal business', 'Leisure [note 2]', 'Just walk [note 8]',
       'Other [note 8]', 'All purposes',
       'Unweighted sample size: trips (thousands)'],
      dtype='object')

In [5]:
nts0409a['Year'].unique()

array([2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
       2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023,
       2024])

In [6]:
nts0409a_processed = nts0409a.copy()
nts0409a_processed = nts0409a_processed[
    nts0409a_processed['Year'] == 2024].copy()

In [7]:
nts0409a_processed

,Year,Main mode,Commuting,Business,Education or escort education,Shopping,Other escort,Personal business,Leisure [note 2],Just walk [note 8],Other [note 8],All purposes,Unweighted sample size: trips (thousands)
308,2024,"Walk [notes 3, 4]",14.090618,2.126124,48.251124,46.972684,8.898228,20.848153,41.463671,84.617743,0.000000,267.268345,82.474
309,2024,Pedal cycle [note 5],4.657803,0.453976,1.397357,1.477198,0.161975,0.799048,6.259976,0.000000,0.000000,15.207333,4.377
310,2024,Car or van driver,60.443130,16.798872,27.906054,76.310214,43.284308,35.432481,96.676435,0.000000,0.501423,357.352918,114.342
311,2024,Car or van passenger,8.244171,1.420307,27.620174,29.601703,21.687584,18.562586,82.233065,0.000000,0.067612,189.437203,57.253
312,2024,Motorcycle,1.037418,0.044498,0.021068,0.207584,0.015937,0.066523,0.597993,0.000000,0.000000,1.991021,0.593
313,2024,Other private transport [note 6],0.512051,0.182144,3.066882,0.782541,0.584211,0.676074,2.255293,0.000000,0.000000,8.059197,2.481
314,2024,Bus in London,2.843678,0.346400,2.445149,2.528714,0.381459,1.364079,3.196108,0.000000,0.009989,13.115577,4.064
315,2024,Other local bus,5.961216,0.562696,5.565624,6.072279,0.999481,2.617272,6.274703,0.000000,0.015282,28.068554,7.749
316,2024,Non-local bus,0.042597,0.000000,0.000000,0.004632,0.000000,0.016646,0.342390,0.000000,0.000000,0.406265,0.120
317,2024,London Underground,4.063161,0.530284,0.697621,0.510131,0.067982,0.559338,3.113857,0.000000,0.000000,9.542374,2.597


In [8]:
# melt means to transform the data from wide format to long format. 
# In this case, we are transforming the data so that each row represents a single trip purpose and its corresponding number of trips for each main mode of transport in the year 2024.
nts0409a_long = nts0409a_processed.melt(
    id_vars=['Year', 'Main mode'],
    value_vars=[col for col in nts0409a_processed.columns if col not in ['Year', 'Main mode']],
    var_name="Purpose",
    value_name="Trips"
)

In [9]:
print(nts0409a_long.head())

   Year             Main mode    Purpose      Trips
0  2024     Walk [notes 3, 4]  Commuting  14.090618
1  2024  Pedal cycle [note 5]  Commuting   4.657803
2  2024     Car or van driver  Commuting  60.443130
3  2024  Car or van passenger  Commuting   8.244171
4  2024            Motorcycle  Commuting   1.037418


In [10]:
nts0409a_long = nts0409a_long[
    ~nts0409a_long['Purpose'].isin([
        'All purposes',
        'Unweighted sample size: trips (thousands)'
    ])
]

nts0409a_long = nts0409a_long[
    nts0409a_long['Main mode'] != 'All modes'
]

nts0409a_long['Probability'] = (
    nts0409a_long['Trips'] /
    nts0409a_long.groupby('Purpose')['Trips'].transform('sum')
)

In [11]:
nts0409a_long

,Year,Main mode,Purpose,Trips,Probability
0,2024,"Walk [notes 3, 4]",Commuting,14.090618,0.126616
1,2024,Pedal cycle [note 5],Commuting,4.657803,0.041854
2,2024,Car or van driver,Commuting,60.443130,0.543133
3,2024,Car or van passenger,Commuting,8.244171,0.074081
4,2024,Motorcycle,Commuting,1.037418,0.009322
5,2024,Other private transport [note 6],Commuting,0.512051,0.004601
6,2024,Bus in London,Commuting,2.843678,0.025553
7,2024,Other local bus,Commuting,5.961216,0.053567
8,2024,Non-local bus,Commuting,0.042597,0.000383
9,2024,London Underground,Commuting,4.063161,0.036511


In [12]:
#verify the sum of probabilities for each purpose equals 1
for purpose in nts0409a_long['Purpose'].unique():
    total_probability = nts0409a_long[nts0409a_long['Purpose'] == purpose]['Probability'].sum()
    print(f"Total probability for {purpose}: {total_probability}")

Total probability for Commuting: 0.9999999999999999
Total probability for Business: 1.0000000000000002
Total probability for Education or escort education: 1.0
Total probability for Shopping: 1.0
Total probability for Other escort: 1.0000000000000002
Total probability for Personal business: 0.9999999999999999
Total probability for Leisure [note 2]: 1.0
Total probability for Just walk [note 8]: 1.0
Total probability for Other [note 8]: 0.9999999999999999


In [13]:
nts0409a_matrix = nts0409a_long.pivot(
    index = 'Purpose',
    columns = 'Main mode',
    values = 'Probability'
)

In [14]:
nts0409a_matrix

Main mode,Bus in London,Car or van driver,Car or van passenger,London Underground,Motorcycle,Non-local bus,Other local bus,Other private transport [note 6],Other public transport [note 7],Pedal cycle [note 5],Surface Rail,Taxi or minicab,"Walk [notes 3, 4]"
Purpose,,,,,,,,,,,,,
Business,0.014149,0.686180,0.058015,0.021660,0.001818,0.000000,0.022984,0.007440,0.002132,0.018543,0.068402,0.011831,0.086845
Commuting,0.025553,0.543133,0.074081,0.036511,0.009322,0.000383,0.053567,0.004601,0.004547,0.041854,0.069340,0.010494,0.126616
Education or escort education,0.020506,0.234031,0.231633,0.005851,0.000177,0.000000,0.046675,0.025720,0.001263,0.011719,0.011596,0.006177,0.404653
Just walk [note 8],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
Leisure [note 2],0.012548,0.379552,0.322847,0.012225,0.002348,0.001344,0.024635,0.008854,0.002579,0.024577,0.029235,0.016470,0.162787
Other [note 8],0.016808,0.843711,0.113766,0.000000,0.000000,0.000000,0.025715,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Other escort,0.004971,0.564046,0.282615,0.000886,0.000208,0.000000,0.013024,0.007613,0.000909,0.002111,0.003210,0.004454,0.115954
Personal business,0.016379,0.425456,0.222890,0.006716,0.000799,0.000200,0.031427,0.008118,0.001792,0.009595,0.011557,0.014737,0.250334
Shopping,0.015155,0.457349,0.177411,0.003057,0.001244,0.000028,0.036393,0.004690,0.001643,0.008853,0.007577,0.005078,0.281521


In [15]:
for purpose in nts0409a_matrix.index:
    total_probability = nts0409a_matrix.loc[purpose].sum()
    print(f"Total probability for {purpose}: {total_probability}")

Total probability for Business: 1.0
Total probability for Commuting: 0.9999999999999999
Total probability for Education or escort education: 1.0
Total probability for Just walk [note 8]: 1.0
Total probability for Leisure [note 2]: 1.0
Total probability for Other [note 8]: 0.9999999999999999
Total probability for Other escort: 1.0000000000000002
Total probability for Personal business: 1.0000000000000002
Total probability for Shopping: 1.0
